# Assignment 3 — ARC-AGI-1 Classical Solver

A small interpretable solver using grid representations, connected objects, candidate transformations, and demonstration-based selection.

## Scope

The assignment asks for a small attempt at exactly one ARC version. This notebook implements a classical,
non-neural ARC-AGI-1 solver. It represents each grid as a matrix of colour integers, extracts connected
objects, proposes a small set of transformations, and ranks candidates using consistency with the
demonstration input/output pairs.

No claim is made that this is a competitive ARC solver. It is deliberately a compact program-search baseline
whose priors and failure modes can be inspected. Supply task JSON files from an official ARC dataset to run
the solver on real tasks.

In [ ]:
import json
import os
from collections import deque, Counter
from copy import deepcopy
import numpy as np

def load_task(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def to_grid(value):
    grid = np.array(value, dtype=int)
    if grid.ndim != 2:
        raise ValueError("An ARC grid must be two-dimensional.")
    if grid.size and (grid.min() < 0 or grid.max() > 9):
        raise ValueError("ARC colours must be integers from 0 to 9.")
    return grid

def task_pairs(task):
    return [(to_grid(pair["input"]), to_grid(pair["output"])) for pair in task["train"]]

def test_inputs(task):
    return [to_grid(item["input"]) for item in task["test"]]

def grid_equal(a, b):
    return a.shape == b.shape and np.array_equal(a, b)

## Connected-component object extraction

In [ ]:
def connected_objects(grid, background=0, diagonal=False):
    rows, cols = grid.shape
    seen = set()
    objects = []
    directions = [(-1,0), (1,0), (0,-1), (0,1)]
    if diagonal:
        directions += [(-1,-1), (-1,1), (1,-1), (1,1)]
    for r in range(rows):
        for c in range(cols):
            colour = int(grid[r, c])
            if colour == background or (r, c) in seen:
                continue
            queue = deque([(r, c)])
            seen.add((r, c))
            cells = []
            while queue:
                x, y = queue.popleft()
                cells.append((x, y))
                for dx, dy in directions:
                    nx, ny = x + dx, y + dy
                    if (0 <= nx < rows and 0 <= ny < cols and
                        (nx, ny) not in seen and int(grid[nx, ny]) == colour):
                        seen.add((nx, ny))
                        queue.append((nx, ny))
            min_r = min(x for x, y in cells)
            max_r = max(x for x, y in cells)
            min_c = min(y for x, y in cells)
            max_c = max(y for x, y in cells)
            objects.append({"colour": colour, "cells": tuple(cells),
                            "bbox": (min_r, min_c, max_r, max_c),
                            "size": len(cells)})
    return objects

def object_summary(grid):
    objects = connected_objects(grid)
    return [{"colour": obj["colour"], "size": obj["size"],
             "bbox": obj["bbox"]} for obj in objects]

## Candidate grid transformations

In [ ]:
def rotate_grid(grid, k=1):
    return np.rot90(grid, k=k).copy()

def reflect_grid(grid, axis="horizontal"):
    if axis == "horizontal":
        return np.fliplr(grid).copy()
    if axis == "vertical":
        return np.flipud(grid).copy()
    raise ValueError("axis must be horizontal or vertical")

def recolour_non_background(grid, new_colour, background=0):
    result = grid.copy()
    result[result != background] = new_colour
    return result

def crop_to_content(grid, background=0):
    points = np.argwhere(grid != background)
    if len(points) == 0:
        return grid.copy()
    r0, c0 = points.min(axis=0)
    r1, c1 = points.max(axis=0) + 1
    return grid[r0:r1, c0:c1].copy()

def transpose_grid(grid):
    return grid.T.copy()

def transform_candidates(grid):
    candidates = []
    for name, fn in (
        ("identity", lambda x: x.copy()),
        ("rotate_90", lambda x: rotate_grid(x, 1)),
        ("rotate_180", lambda x: rotate_grid(x, 2)),
        ("rotate_270", lambda x: rotate_grid(x, 3)),
        ("reflect_horizontal", lambda x: reflect_grid(x, "horizontal")),
        ("reflect_vertical", lambda x: reflect_grid(x, "vertical")),
        ("transpose", transpose_grid),
        ("crop_to_content", crop_to_content)
    ):
        candidates.append((name, fn(grid)))
    non_background = sorted(int(x) for x in np.unique(grid) if x != 0)
    for colour in range(1, 10):
        candidates.append((f"recolour_to_{colour}", recolour_non_background(grid, colour)))
    return candidates

## Learn a transformation from demonstrations

In [ ]:
def candidate_score(pairs, transform):
    score = 0
    for source, target in pairs:
        prediction = transform(source)
        if grid_equal(prediction, target):
            score += 1
        else:
            if prediction.shape == target.shape:
                score += float(np.mean(prediction == target))
    return score / max(1, len(pairs))

def infer_rule(pairs):
    candidates = []
    names = [
        ("identity", lambda x: x.copy()),
        ("rotate_90", lambda x: rotate_grid(x, 1)),
        ("rotate_180", lambda x: rotate_grid(x, 2)),
        ("rotate_270", lambda x: rotate_grid(x, 3)),
        ("reflect_horizontal", lambda x: reflect_grid(x, "horizontal")),
        ("reflect_vertical", lambda x: reflect_grid(x, "vertical")),
        ("transpose", transpose_grid),
        ("crop_to_content", crop_to_content)
    ]
    for name, fn in names:
        candidates.append((candidate_score(pairs, fn), name, fn))
    for colour in range(1, 10):
        fn = lambda x, colour=colour: recolour_non_background(x, colour)
        candidates.append((candidate_score(pairs, fn), f"recolour_to_{colour}", fn))
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates

def solve_task(task):
    pairs = task_pairs(task)
    ranked = infer_rule(pairs)
    predictions = []
    for test_grid in test_inputs(task):
        valid = []
        for score, name, transform in ranked:
            try:
                output = transform(test_grid)
                if output.ndim == 2 and output.size <= 900:
                    valid.append({"score": score, "rule": name, "output": output})
            except Exception:
                pass
        predictions.append(valid[:3])
    return ranked, predictions

## Local tests with small constructed examples

In [ ]:
toy_task = {
    "train": [
        {"input": [[0, 1, 0], [0, 0, 0]], "output": [[0, 0], [1, 0], [0, 0]]},
        {"input": [[0, 2, 0], [0, 0, 0]], "output": [[0, 0], [2, 0], [0, 0]]}
    ],
    "test": [{"input": [[0, 3, 0], [0, 0, 0]]}]
}

toy_pairs = task_pairs(toy_task)
ranked, predictions = solve_task(toy_task)
print("Top inferred rules:", [(round(score, 3), name) for score, name, _ in ranked[:5]])
for candidate in predictions[0]:
    print(candidate["rule"], candidate["output"].tolist())

assert connected_objects(to_grid([[0, 1], [0, 1]]))[0]["size"] == 2
assert grid_equal(rotate_grid(to_grid([[1, 2], [3, 4]])),
                  to_grid([[2, 4], [1, 3]]))

## Evaluate on a folder of official ARC task JSON files

In [ ]:
def evaluate_folder(folder, limit=None):
    paths = sorted(os.path.join(folder, name) for name in os.listdir(folder)
                   if name.endswith(".json"))
    if limit is not None:
        paths = paths[:limit]
    records = []
    for path in paths:
        task = load_task(path)
        ranked, predictions = solve_task(task)
        exact = 0
        total = 0
        for test_item, candidates in zip(task["test"], predictions):
            expected = to_grid(test_item["output"]) if "output" in test_item else None
            if expected is None:
                continue
            total += 1
            if candidates and grid_equal(candidates[0]["output"], expected):
                exact += 1
        records.append({"task": os.path.basename(path), "scored_tests": total,
                        "exact": exact, "accuracy": exact / total if total else None,
                        "best_rule": ranked[0][1]})
    return records

print("Place official ARC task JSON files in a folder, then call evaluate_folder(path).")

## Failure analysis and scaling

This baseline can solve only tasks expressible by a small fixed set of global transformations.
It will fail when a task requires object counting, conditional rules, multi-object relations, drawing,
symmetry completion, colour mapping that depends on object identity, or a composition of several operations.
A better classical solver would build a domain-specific language of primitives, enumerate short programs,
and use all demonstrations to reject inconsistent programs.

The hard-coded priors are: background colour 0, four-neighbour connectivity, and the assumption that a
single simple transformation applies to every example. These priors reduce search but can be wrong.
For larger DSLs, program search grows combinatorially with program length and branching factor.

The assignment's ARC-AGI-1 grid constraints are up to 30 by 30 with ten colours. The solver validates
colour range and matrix dimensionality, but does not claim broad ARC competence. A failed task is useful
evidence about the limits of the chosen hypothesis language.